# Songdo Urban Digital Memory — Master Google Colab Starter Kit
### Database Management and Practice — Prof. Mehdi Pirahandeh, Inha University
**Goal:** working mini-platform + teaching lab + five-team starter repository.

## MODULE 00 — Welcome & Architecture
**Write path:** source → adapter → validation → deduplication → domain insert/quarantine.  
**Read path:** PostgreSQL/PostGIS → indexed SQL/views → FastAPI → consumers.
```text
Physical/Open Data → Adapters → Ingestion → PostgreSQL/PostGIS → Historical Ledger → Query/API → Consumers
```

## MODULE 01 — Colab Environment Setup
**Purpose:** diagnose Python, OS, RAM, disk, psql and packages.
**Expected:** PASS/WARNING table. **Try:** add a Uvicorn check. **Connection:** Week 1 reproducibility.

In [ ]:
import os,sys,shutil,platform,importlib.util,pandas as pd
from pathlib import Path
checks=[]
def add(c,d,s): checks.append({"Component":c,"Detail":d,"Status":s})
add("Python",sys.version.split()[0],"PASS"); add("OS",platform.platform(),"PASS"); total,used,free=shutil.disk_usage("/"); add("Disk",f"{free/1024**3:.1f} GB free","PASS" if free>2*1024**3 else "WARNING"); add("psql",shutil.which("psql") or "not found","PASS" if shutil.which("psql") else "WARNING")
for p in ["psycopg","fastapi","pydantic","pandas","geopandas","shapely","httpx"]: add(p,"installed" if importlib.util.find_spec(p) else "not installed","PASS" if importlib.util.find_spec(p) else "WARNING")
display(pd.DataFrame(checks))

## MODULE 02 — PostgreSQL + PostGIS Installation
Colab-native installation; Docker is not required. The password is local to the temporary runtime only.

In [ ]:
import subprocess,shutil,os
def run(cmd): print("+",cmd); subprocess.run(cmd,shell=True,check=True)
if shutil.which("psql") is None: run("apt-get update -qq"); run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql postgresql-contrib postgis")
run("service postgresql start"); run("pg_isready")
DB_NAME="songdo_urban_memory"; DB_USER="songdo_admin"; DB_PASSWORD="songdo_colab_local_only"
run(f"sudo -u postgres psql -v ON_ERROR_STOP=1 -c \"DO \\$\\$ BEGIN IF NOT EXISTS (SELECT FROM pg_roles WHERE rolname='{DB_USER}') THEN CREATE ROLE {DB_USER} LOGIN PASSWORD '{DB_PASSWORD}'; END IF; END \\$\\$;\"")
exists=subprocess.run(f"sudo -u postgres psql -tAc \"SELECT 1 FROM pg_database WHERE datname='{DB_NAME}'\"",shell=True,text=True,capture_output=True).stdout.strip()
if exists!="1": run(f"sudo -u postgres createdb -O {DB_USER} {DB_NAME}")
os.environ.update(DB_NAME=DB_NAME,DB_USER=DB_USER,DB_PASSWORD=DB_PASSWORD,DB_HOST="127.0.0.1",DB_PORT="5432")
run(f"sudo -u postgres psql -d {DB_NAME} -c \"CREATE EXTENSION IF NOT EXISTS postgis; CREATE EXTENSION IF NOT EXISTS btree_gist; SELECT PostGIS_Full_Version();\"")

## MODULE 03 — Repository Generation
Upload the instructor ZIP below, then extract it to `/content`.

In [ ]:
from google.colab import files
import zipfile,os
from pathlib import Path
uploaded=files.upload(); zname=next(n for n in uploaded if n.endswith(".zip")); zipfile.ZipFile(zname).extractall("/content"); PROJECT=Path("/content/Songdo_Urban_Data_Platform"); os.chdir(PROJECT); print(PROJECT)

## MODULE 04 — Database Initialization
Apply all 8 migrations in dependency order, then deterministic starter fixtures.

In [ ]:
import subprocess,os
env={**os.environ,"PGPASSWORD":DB_PASSWORD}
for f in ["sql/00_init_database.sql","sql/01_core_schema.sql","sql/02_db1_mobility.sql","sql/03_db2_building.sql","sql/04_db3_tourism.sql","sql/05_db4_environment.sql","sql/06_db5_safety.sql","sql/07_cross_domain_views.sql"]: print("Applying",f); subprocess.run(["psql","-h","127.0.0.1","-U",DB_USER,"-d",DB_NAME,"-f",f],env=env,check=True)
subprocess.run(["psql","-h","127.0.0.1","-U",DB_USER,"-d",DB_NAME,"-f","fixtures/seed_data.sql"],env=env,check=True)

## MODULE 05 — Core Shared Schema
Inspect core tables and verify boundary logic.

In [ ]:
subprocess.run(["psql","-h","127.0.0.1","-U",DB_USER,"-d",DB_NAME,"-c","SELECT table_schema,table_name FROM information_schema.tables WHERE table_schema IN ('core','mobility','building','tourism','environment','safety','cross_domain') ORDER BY 1,2; SELECT core.is_in_songdo(ST_SetSRID(ST_MakePoint(126.6548,37.3888),4326));"],env=env,check=True)

## MODULE 06 — Data Source Registry
Levels: 1 metadata, 2 docs/schema, 3 real payload/API/file, 4 Songdo subset. Synthetic fixtures are never official data.

In [ ]:
display(pd.read_csv("source_registry.csv"))

## MODULE 07 — Data Quality Pipeline
Test valid, outside, swapped/malformed, missing coordinates and SHA-256 idempotency.

In [ ]:
from adapters.base_adapter import validate_coordinate,calculate_payload_hash
s=[{"case":"valid","lon":126.6548,"lat":37.3888},{"case":"outside","lon":127.0,"lat":37.5},{"case":"swapped","lon":37.3888,"lat":126.6548},{"case":"malformed","lon":"ABC","lat":37.38},{"case":"missing","lon":None,"lat":None}]
display(pd.DataFrame([{**x,"result":validate_coordinate(x["lon"],x["lat"])} for x in s])); print(calculate_payload_hash({"vehicle":"V001","speed":20}))

## MODULE 08 — DB1 Mobility
DB1-1 ordered route; DB1-2 lateness; DB1-3 historical replay; DB1-4 LAG/LEAD; DB1-5 spatial telemetry.

In [ ]:
import psycopg,pandas as pd
conn=psycopg.connect(f"dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host=127.0.0.1")
qs=["SELECT rs.route_id,rs.direction,rs.sequence_order,s.stop_name_en FROM mobility.route_stops rs JOIN mobility.stops s USING(stop_id) ORDER BY 1,2,3","SELECT trip_id,stop_id,scheduled_arrival,observed_arrival,lateness_seconds FROM mobility.trip_stop_events ORDER BY scheduled_arrival","SELECT DISTINCT ON(vehicle_id) vehicle_id,ST_X(geom) lon,ST_Y(geom) lat,speed_kmh,observed_at FROM mobility.vehicle_observations WHERE observed_at<='2026-10-06 08:06+09' ORDER BY vehicle_id,observed_at DESC","SELECT trip_id,stop_id,observed_arrival,LAG(observed_arrival) OVER(ORDER BY observed_arrival) previous_arrival FROM mobility.trip_stop_events","SELECT stop_id,stop_name_en,round(ST_Distance(geom::geography,ST_SetSRID(ST_MakePoint(126.646,37.388),4326)::geography)::numeric,1) distance_m FROM mobility.stops WHERE ST_DWithin(geom::geography,ST_SetSRID(ST_MakePoint(126.646,37.388),4326)::geography,1000) ORDER BY distance_m"]
for q in qs: display(pd.read_sql(q,conn))

## MODULE 09 — DB2 Smart Building

In [ ]:
for q in ["SELECT b.building_name_en,sp.space_name,d.device_id,d.device_type FROM building.buildings b JOIN building.spaces sp ON sp.building_id=b.building_id JOIN building.device_space_assignments a ON a.space_id=sp.space_id AND a.unassigned_at IS NULL JOIN building.devices d ON d.device_id=a.device_id ORDER BY d.device_id","SELECT d.device_id FROM building.devices d WHERE NOT EXISTS(SELECT 1 FROM building.telemetry t WHERE t.device_id=d.device_id AND t.observed_at>clock_timestamp()-interval '15 minutes')","SELECT * FROM building.telemetry ORDER BY observed_at DESC"]: display(pd.read_sql(q,conn))

## MODULE 10 — DB3 Tourism

In [ ]:
display(pd.read_sql("SELECT a.attraction_name_en,s.stop_name_en,round(ST_Distance(a.geom::geography,s.geom::geography)::numeric,1) distance_m FROM tourism.attractions a JOIN mobility.stops s ON ST_DWithin(a.geom::geography,s.geom::geography,1000) ORDER BY distance_m",conn)); print("Starter Demand Estimation Example — educational baseline, not production AI.")

## MODULE 11 — DB4 Environment

In [ ]:
display(pd.read_sql("SELECT DISTINCT ON(station_id,type_code) station_id,type_code,measured_value,quality_flag,observed_at FROM environment.measurements ORDER BY station_id,type_code,observed_at DESC",conn)); display(pd.read_sql("SELECT station_id,type_code,date_trunc('hour',observed_at) hour_start,avg(measured_value),min(measured_value),max(measured_value),count(*) FROM environment.measurements WHERE quality_flag='GOOD' GROUP BY 1,2,3 ORDER BY 3,2",conn))

## MODULE 12 — DB5 Safety

In [ ]:
display(pd.read_sql("SELECT incident_id,title,severity,current_status,reported_at FROM safety.incidents",conn)); display(pd.read_sql("SELECT safety.valid_incident_transition('REPORTED','DISPATCHED') allowed_1,safety.valid_incident_transition('REPORTED','RESOLVED') allowed_2",conn))

## MODULE 13 — Historical Urban Memory
Current state ≠ historical state ≠ audit event.

In [ ]:
def get_vehicle_state_as_of(t): return pd.read_sql("SELECT DISTINCT ON(vehicle_id) vehicle_id,ST_X(geom) lon,ST_Y(geom) lat,speed_kmh,observed_at FROM mobility.vehicle_observations WHERE observed_at<=%s::timestamptz ORDER BY vehicle_id,observed_at DESC",conn,params=(t,))
display(get_vehicle_state_as_of("2026-10-06 08:06+09"))

## MODULE 14 — PostGIS Teaching Lab
Demonstrates ST_Point/ST_SetSRID/ST_Distance/ST_DWithin/ST_X/ST_Y/ST_AsGeoJSON and geometry vs geography.

In [ ]:
display(pd.read_sql("SELECT ST_AsText(ST_SetSRID(ST_MakePoint(126.646,37.388),4326)) wkt,round(ST_Distance(ST_SetSRID(ST_MakePoint(126.646,37.388),4326)::geography,ST_SetSRID(ST_MakePoint(126.6375,37.3920),4326)::geography)::numeric,1) distance_m,ST_AsGeoJSON(ST_SetSRID(ST_MakePoint(126.646,37.388),4326)) geojson",conn))

## MODULE 15 — Cross-Team Integration

In [ ]:
for v in ["cross_domain.v_attractions_events_near_stops","cross_domain.v_building_devices_with_environmental_context","cross_domain.v_incidents_mobility_environmental_impact"]: print(v); display(pd.read_sql(f"SELECT * FROM {v} LIMIT 20",conn))

## MODULE 16 — FastAPI
Starts Uvicorn locally; no public tunnel required.

In [ ]:
import subprocess,time,requests,sys,os
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True); subprocess.run("pkill -f 'uvicorn api.main:app' || true",shell=True); log=open("outputs/logs/uvicorn.log","w"); api_proc=subprocess.Popen([sys.executable,"-m","uvicorn","api.main:app","--host","127.0.0.1","--port","8000"],stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy()); time.sleep(3); r=requests.get("http://127.0.0.1:8000/health",timeout=5); print(r.status_code,r.json())

## MODULE 17 — Downstream Consumers
ROS-like telemetry simulator, VIP indoor/outdoor client, digital-twin replay.

In [ ]:
from consumers.vip_building_env_client import get_context
print(get_context())
from consumers.digital_twin_temporal_replay import vehicle_state_as_of
print(vehicle_state_as_of("2026-10-06 08:06+09"))

## MODULE 18 — Testing
PASS is based on actual pytest result.

In [ ]:
os.environ["SONGDO_DB_TESTS"]="1"; test_run=subprocess.run([sys.executable,"-m","pytest","-q"],text=True,capture_output=True); print(test_run.stdout); print(test_run.stderr); Path("outputs/evidence/test_results.txt").write_text(test_run.stdout+"\n"+test_run.stderr,encoding="utf-8")

## MODULE 19 — Performance Engineering
Inspect the actual plan; do not fabricate latency.

In [ ]:
with conn.cursor() as cur: cur.execute("EXPLAIN (ANALYZE,BUFFERS) SELECT stop_id FROM mobility.stops WHERE ST_DWithin(geom::geography,ST_SetSRID(ST_MakePoint(126.646,37.388),4326)::geography,1000)"); print("\n".join(r[0] for r in cur.fetchall()))

## MODULE 20 — Backup / Export

In [ ]:
backup="outputs/songdo_urban_memory.backup"; subprocess.run(["pg_dump","-h","127.0.0.1","-U",DB_USER,"-d",DB_NAME,"-Fc","-f",backup],env={**os.environ,"PGPASSWORD":DB_PASSWORD},check=True); print(Path(backup).stat().st_size,"bytes")

## MODULE 21 — Team Handoff
DB1→`02_db1_mobility.sql`; DB2→`03_db2_building.sql`; DB3→`04_db3_tourism.sql`; DB4→`05_db4_environment.sql`; DB5→`06_db5_safety.sql`.  
START → EXTEND TABLES → FIVE SCENARIOS → QUERIES/API → TESTS → EVIDENCE.

## MODULE 22 — Final Verification Dashboard
Statuses are computed from runtime checks.

In [ ]:
def sql_ok(q):
 try:
  with conn.cursor() as cur: cur.execute(q); cur.fetchall(); return True
 except Exception: conn.rollback(); return False
checks=[("PostgreSQL","Connection",sql_ok("SELECT 1")),("PostGIS","Extension",sql_ok("SELECT PostGIS_Full_Version()")),("Core","Tables",sql_ok("SELECT place_id FROM core.places LIMIT 1")),("DB1","Mobility",sql_ok("SELECT vehicle_id FROM mobility.vehicles LIMIT 1")),("DB2","Building",sql_ok("SELECT building_id FROM building.buildings LIMIT 1")),("DB3","Tourism",sql_ok("SELECT attraction_id FROM tourism.attractions LIMIT 1")),("DB4","Environment",sql_ok("SELECT station_id FROM environment.stations LIMIT 1")),("DB5","Safety",sql_ok("SELECT incident_id FROM safety.incidents LIMIT 1")),("Cross-domain","Views",sql_ok("SELECT * FROM cross_domain.v_attractions_events_near_stops LIMIT 1")),("Tests","pytest",test_run.returncode==0),("Backup","File",Path("outputs/songdo_urban_memory.backup").exists())]
df=pd.DataFrame([{"Component":a,"Verification":b,"Status":"PASS" if c else "FAIL"} for a,b,c in checks]); display(df); df.to_csv("outputs/evidence/verification_summary.csv",index=False)

# YOU NOW HAVE
PostgreSQL + PostGIS + core + five domain schemas + fixtures + data-quality foundation + historical ledger + spatial/temporal SQL + FastAPI + cross-domain views + consumers + tests + backup + evidence.

> This is the beginning, not the final submission. Each team must extend, validate official sources, complete all five scenarios, benchmark, document, and defend its work.